# Recovered RL Project Notebook

Recovered after the power interruption from IPython history and the existing project source files.

**Important:** The recovered `%%writefile` cells are preserved for reference. Do not execute them unless you intentionally want to rewrite/append to the corresponding `.py` files.

## TileCoder.indices

In [ ]:
%%writefile -a src/tilecoding.py

    def indices(self, state):
        """Return the `num_tilings` active feature indices for `state`"""

        s = np.clip(np.asarray(state, dtype=np.float64), self.low, self.high)
        q = np.floor((s-self.low) * self.scale * self.num_tilings).astype(np.int64)

        coords = (q[None, :] + self._offsets) // self.num_tilings

        out = np.empty(self.num_tilings, dtype=np.int64)
        get = self.iht.get_index
        for t in range(self.num_tilings):
            out[t] = get(hash((t, *coords[t].tolist())))
        return out

## default_tiling_config

In [ ]:
%%writefile -a src/tilecoding.py

def default_tiling_config(env_id):
    cfg = {
        "MountainCar-v0": dict(num_tilings=8, tiles_per_dim=8, memory_size=4096),
        "CartPole-v1": dict(num_tilings=8, tiles_per_dim=6, memory_size=32768),
        "Acrobot-v1": dict(num_tilings=8, tiles_per_dim=6, memory_size=131072),
        "LunarLander-v3": dict(num_tilings=16, tiles_per_dim=4, memory_size=262144),
    }

    return dict(cfg[env_id])

## argmax_random_tie

In [ ]:
%%writefile src/exploration.py

import numpy as np

def argmax_random_tie(q, rng):
    """Argmax with ties broken uniformly at random"""

    m = q.max()
    ties = np.flatnonzero(q == m)
    if ties.size == 1:
        return int(ties[0])
    return int(ties[rng.integers(ties.size)])

## Explorer

In [ ]:
%%writefile -a src/exploration.py

class Explorer:
    name = "base"
    uses_features = False

    def __init__(self, n_actions):
        self.n_actions = int(n_actions)
        self.t = 0
        self.current_epsilon = 0.0

    def _epsilon_now(self, feat_idx=None):
        raise NotImplementedError
    
    def update(self, td_error, value=0.0, feat_idx=None):
        return None
    
    def reset_episode(self):
        return None
    
    def select(self, q_values, rng, feat_idx=None):
        eps = self._epsilon_now(feat_idx)
        self.current_epsilon = eps
        self.t += 1
        if rng.random() < eps:
            return int(rng.integers(self.n_actions))
        return argmax_random_tie(q_values, rng)

## FixedEpsilon

In [ ]:
%%writefile -a src/exploration.py

class FixedEpsilon(Explorer):
    name = "fixed"
    
    def __init__(self, n_actions, epsilon=0.1):
        super().__init__(n_actions)
        self.epsilon = float(epsilon)

    def _epsilon_now(self, feat_idx=None):
        return self.epsilon

## DecayEpsilon

In [ ]:
%%writefile -a src/exploration.py

class DecayEpsilon(Explorer):
    name = "decay"

    def __init__(self, n_actions, eps_start=1, eps_end=0.01, decay_steps=50_000, mode="exponential"):
        super().__init__(n_actions)
        self.eps_start, self.eps_end = float(eps_start), float(eps_end)
        self.decay_steps, self.mode = int(decay_steps), mode
        self._rate = np.log(max(eps_end, 1e-12)/eps_start)/max(decay_steps, 1)

## Boltzmann

In [ ]:
%%writefile -a src/exploration.py

class Boltzmann(Explorer):
    name = "boltzmann"

    def __init__(self, n_actions, tau_start=1.0, tau_end=0.05, decay_steps=50_000):
        super().__init__(n_actions)
        self.tau_start, self.tau_end = float(tau_start), float(tau_end)
        self.decay_steps = int(decay_steps)
        self._rate = np.log(max(tau_end, 1e-12)/tau_start)/max(decay_steps, 1)

    def select(self, q_values, rng, feat_idx=None):
        tau = float(max(self.tau_end, self.tau_start * np.exp(self._rate * self.t)))
        self.t += 1
        z = np.clip((q_values - q_values.max()) / max(tau, 1e-8), -50.0, 0.0)
        p = np.exp(z)
        p /= p.sum()
        self.current_epsilon = float(1.0 -p[int(np.argmax(q_values))])
        return int(rng.choice(self.n_actions, p=p))

## VDBE

In [ ]:
%%writefile -a src/exploration.py

class VDBE(Explorer):
    name = "vdbe"

    def __init__(self, n_actions, sigma=1.0, eps_init=1.0, alpha_scale=1.0, eps_min=0.0):
        super().__init__(n_actions)
        self.sigma = float(sigma)
        self.eps = float(eps_init)
        self.delta_param = 1.0 / self.n_actions
        self.alpha_scale = float(alpha_scale)
        self.eps_min = float(eps_min)

    def _epsilon_now(self, feat_idx=None):
        return max(self.eps_min, self.eps)

    @staticmethod
    def _f(x):
        e = np.exp(-min(x, 50.0))
        return float((1.0 - e) / (1.0 + e))
    
    def update(self, td_error, value=0.0, feat_idx=None):
        f = self._f(abs(self.alpha_scale * float(td_error)) / max(self.sigma, 1e-12))
        d = self.delta_param
        self.eps = d * f + (1.0 - d) * self.eps

## RATE

In [ ]:
%%writefile -a src/exploration.py

class RATE(Explorer):
    name = "rate"

    def __init__(self, n_actions, eps_min=0.01, eps_max=1.0, beta=0.01, kappa=1.0):
        super().__init__(n_actions)
        self.eps_min, self.eps_max = float(eps_min), float(eps_max)
        self.beta, self.kappa = float(beta), float(kappa)
        self.m_d = 0.0
        self.m_v = 0.0
        self.rho = 1.0

    def _epsilon_now(self, feat_idx=None):
        return self.eps_min + (self.eps_max - self.eps_min) * (self.rho ** self.kappa)
    
    def update(self, td_error, value=0.0, feat_idx=None):
        b = self.beta
        self.m_d += b * (abs(float(td_error)) - self.m_d)
        self.m_v += b * (abs(float(value)) - self.m_v)
        den = self.m_d + self.m_v
        self.rho = (self.m_d / den) if den > 1e-8 else 1.0